In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%cd /content/drive/MyDrive/Colab\ Notebooks/ampi

/content/drive/MyDrive/Colab Notebooks/ampi


In [3]:
%%writefile ampi_index.cpp

// ampi_index.cpp
#include <pybind11/pybind11.h>
#include <pybind11/numpy.h>
#include <pybind11/stl.h>

#include <vector>
#include <algorithm>
#include <cmath>
#include <numeric>
#include <random>
#ifdef _OPENMP
#include <omp.h>
#endif

namespace py = pybind11;

// Helper: Convert a 2D numpy array (float32) to a std::vector<std::vector<float>>
std::vector<std::vector<float>> numpy_to_vector2d(py::array_t<float> arr) {
    py::buffer_info buf = arr.request();
    if (buf.ndim != 2)
        throw std::runtime_error("Input should be a 2-D numpy array");
    int n = buf.shape[0];
    int d = buf.shape[1];
    float* ptr = static_cast<float*>(buf.ptr);
    std::vector<std::vector<float>> vec(n, std::vector<float>(d));
    for (int i = 0; i < n; i++) {
        for (int j = 0; j < d; j++) {
            vec[i][j] = ptr[i * d + j];
        }
    }
    return vec;
}

// Helper: Convert a 1D numpy array (float32) to std::vector<float>
std::vector<float> numpy_to_vector1d(py::array_t<float> arr) {
    py::buffer_info buf = arr.request();
    if (buf.ndim != 1)
        throw std::runtime_error("Input should be a 1-D numpy array");
    int n = buf.shape[0];
    float* ptr = static_cast<float*>(buf.ptr);
    std::vector<float> vec(n);
    for (int i = 0; i < n; i++) {
        vec[i] = ptr[i];
    }
    return vec;
}

// Helper: Convert a std::vector<std::vector<float>> to a 2D numpy array (float32)
py::array_t<float> vector2d_to_numpy(const std::vector<std::vector<float>>& vec) {
    if (vec.empty())
        return py::array_t<float>();
    size_t k = vec.size();
    size_t d = vec[0].size();
    std::vector<ssize_t> shape = { static_cast<ssize_t>(k), static_cast<ssize_t>(d) };
    py::array_t<float> arr(shape);
    py::buffer_info buf = arr.request();
    float* ptr = static_cast<float*>(buf.ptr);
    for (size_t i = 0; i < k; i++) {
        for (size_t j = 0; j < d; j++) {
            ptr[i * d + j] = vec[i][j];
        }
    }
    return arr;
}

// AMPIIndex: Adaptive Multi-Projection Index
class AMPIIndex {
public:
    // Declare data first, then other members to avoid reordering warnings.
    std::vector<std::vector<float>> data;         // Data: (n x d)
    int n, d, L;
    std::vector<std::vector<float>> proj_dirs;      // Projection directions: (L x d)
    std::vector<std::vector<float>> sorted_projs;   // Sorted projection values per direction: (L x n)
    std::vector<std::vector<int>> sorted_idxs;      // Corresponding sorted indices: (L x n)

    // Constructor: build index from data and number of projections.
    AMPIIndex(const std::vector<std::vector<float>>& data_in, int num_projections) {
        data = data_in;
        n = data.size();
        d = (n > 0) ? data[0].size() : 0;
        L = num_projections;

        // Initialize random generator with seed 0.
        std::mt19937 rng(0);
        std::normal_distribution<float> dist(0.0f, 1.0f);

        // Generate L random projection directions and normalize each.
        proj_dirs.resize(L, std::vector<float>(d, 0.0f));
        for (int i = 0; i < L; i++) {
            float norm = 0.0f;
            for (int j = 0; j < d; j++) {
                proj_dirs[i][j] = dist(rng);
                norm += proj_dirs[i][j] * proj_dirs[i][j];
            }
            norm = std::sqrt(norm);
            for (int j = 0; j < d; j++) {
                proj_dirs[i][j] /= norm;
            }
        }

        // Precompute projections for each direction.
        sorted_projs.resize(L, std::vector<float>(n, 0.0f));
        sorted_idxs.resize(L, std::vector<int>(n, 0));
        for (int i = 0; i < L; i++) {
            std::vector<float> projs(n, 0.0f);
            std::vector<int> idxs(n);
            for (int k = 0; k < n; k++) {
                float dot = 0.0f;
                for (int j = 0; j < d; j++) {
                    dot += data[k][j] * proj_dirs[i][j];
                }
                projs[k] = dot;
                idxs[k] = k;
            }
            // Sort indices by corresponding projection values.
            std::vector<int> sorted_indices = idxs;
            std::sort(sorted_indices.begin(), sorted_indices.end(), [&projs](int a, int b) {
                return projs[a] < projs[b];
            });
            // Save sorted projections and indices.
            for (int k = 0; k < n; k++) {
                sorted_projs[i][k] = projs[sorted_indices[k]];
                sorted_idxs[i][k] = sorted_indices[k];
            }
        }
    }

    // Extract candidate indices from projections given a query vector q.
    std::vector<int> query_candidates(const std::vector<float>& q, int window_size = 10) {
        // Preallocate candidate matrix (L x (2*window_size))
        std::vector<std::vector<int>> candidate_matrix(L, std::vector<int>(2 * window_size, 0));

        // Loop over each projection direction (parallelized if OpenMP is enabled)
#ifdef _OPENMP
        #pragma omp parallel for
#endif
        for (int i = 0; i < L; i++) {
            float q_val = 0.0f;
            for (int j = 0; j < d; j++) {
                q_val += proj_dirs[i][j] * q[j];
            }
            // Binary search in sorted_projs[i]
            auto& proj = sorted_projs[i];
            int idx = std::lower_bound(proj.begin(), proj.end(), q_val) - proj.begin();
            int slice_size = 2 * window_size;
            int start = idx - window_size;
            if (start < 0)
                start = 0;
            if (start > n - slice_size)
                start = n - slice_size;
            // Extract candidate indices from sorted_idxs[i]
            for (int k = 0; k < slice_size; k++) {
                candidate_matrix[i][k] = sorted_idxs[i][start + k];
            }
        }
        // Flatten the candidate matrix.
        std::vector<int> flat;
        flat.reserve(L * 2 * window_size);
        for (int i = 0; i < L; i++) {
            for (int j = 0; j < 2 * window_size; j++) {
                flat.push_back(candidate_matrix[i][j]);
            }
        }
        // Sort and remove duplicates.
        std::sort(flat.begin(), flat.end());
        auto last = std::unique(flat.begin(), flat.end());
        flat.erase(last, flat.end());
        return flat;
    }

    // Compute squared Euclidean distances between query q and candidate points.
    std::vector<float> compute_dists(const std::vector<float>& q, const std::vector<int>& candidate_indices) {
        std::vector<float> dists(candidate_indices.size(), 0.0f);
        for (size_t i = 0; i < candidate_indices.size(); i++) {
            float s = 0.0f;
            int idx = candidate_indices[i];
            for (int j = 0; j < d; j++) {
                float diff = data[idx][j] - q[j];
                s += diff * diff;
            }
            dists[i] = s;
        }
        return dists;
    }

    // Query for k nearest neighbors.
    // Returns final_points (k x d), final_dists (k), and final_indices.
    void query(const std::vector<float>& q, int window_size, int k,
               std::vector<std::vector<float>>& final_points,
               std::vector<float>& final_dists,
               std::vector<int>& final_indices) {
        std::vector<int> candidate_indices = query_candidates(q, window_size);
        std::vector<float> dists = compute_dists(q, candidate_indices);
        std::vector<int> order(candidate_indices.size());
        std::iota(order.begin(), order.end(), 0);
        std::sort(order.begin(), order.end(), [&dists](int a, int b) {
            return dists[a] < dists[b];
        });
        final_points.clear();
        final_dists.clear();
        final_indices.clear();
        for (int i = 0; i < k && i < (int)order.size(); i++) {
            int idx = candidate_indices[order[i]];
            final_points.push_back(data[idx]);
            final_dists.push_back(dists[order[i]]);
            final_indices.push_back(idx);
        }
    }

    // ---- Python Wrapper Methods ----

    // Wrapper for query_candidates: accepts a 1D numpy array for query.
    std::vector<int> py_query_candidates(py::array_t<float> q, int window_size = 10) {
        std::vector<float> q_vec = numpy_to_vector1d(q);
        return query_candidates(q_vec, window_size);
    }

    // Wrapper for query: accepts a 1D numpy array and returns a tuple
    // (final_points as a 2D numpy array, final_dists as a list, final_indices as a list).
    py::tuple py_query(py::array_t<float> q, int window_size, int k) {
        std::vector<float> q_vec = numpy_to_vector1d(q);
        std::vector<std::vector<float>> final_points;
        std::vector<float> final_dists;
        std::vector<int> final_indices;
        query(q_vec, window_size, k, final_points, final_dists, final_indices);
        py::array_t<float> np_points = vector2d_to_numpy(final_points);
        return py::make_tuple(np_points, final_dists, final_indices);
    }
};

PYBIND11_MODULE(ampi_index, m) {
    m.doc() = "AMPIIndex module compiled with C++ and pybind11";
    // Use a lambda in the constructor to convert numpy array to std::vector<std::vector<float>>
    m.def("create_ampi_index", [](py::array_t<float> data, int num_projections) {
        auto vec = numpy_to_vector2d(data);
        return new AMPIIndex(vec, num_projections);
    }, py::return_value_policy::take_ownership, py::arg("data"), py::arg("num_projections"));

    py::class_<AMPIIndex>(m, "AMPIIndex")
        // Remove the direct constructor binding and use our factory function instead.
        .def("query_candidates", &AMPIIndex::py_query_candidates, py::arg("q"), py::arg("window_size") = 10)
        .def("query", &AMPIIndex::py_query, py::arg("q"), py::arg("window_size"), py::arg("k"));
}

Overwriting ampi_index.cpp


In [4]:
%%writefile ampi_index.cpp

// ampi_index.cpp
#include <pybind11/pybind11.h>
#include <pybind11/numpy.h>
#include <pybind11/stl.h>
#include <vector>
#include <algorithm>
#include <cmath>
#include <numeric>
#include <random>
#ifdef _OPENMP
#include <omp.h>
#endif

// Include the BLAS header.
#include <cblas.h>

namespace py = pybind11;

// Helper: Convert a 2D numpy array (float32) to a std::vector<std::vector<float>>
std::vector<std::vector<float>> numpy_to_vector2d(py::array_t<float> arr) {
    py::buffer_info buf = arr.request();
    if (buf.ndim != 2)
        throw std::runtime_error("Input should be a 2-D numpy array");
    int n = buf.shape[0];
    int d = buf.shape[1];
    float* ptr = static_cast<float*>(buf.ptr);
    std::vector<std::vector<float>> vec(n, std::vector<float>(d));
    for (int i = 0; i < n; i++) {
        for (int j = 0; j < d; j++) {
            vec[i][j] = ptr[i * d + j];
        }
    }
    return vec;
}

// Helper: Convert a 1D numpy array (float32) to std::vector<float>
std::vector<float> numpy_to_vector1d(py::array_t<float> arr) {
    py::buffer_info buf = arr.request();
    if (buf.ndim != 1)
        throw std::runtime_error("Input should be a 1-D numpy array");
    int n = buf.shape[0];
    float* ptr = static_cast<float*>(buf.ptr);
    std::vector<float> vec(n);
    for (int i = 0; i < n; i++) {
        vec[i] = ptr[i];
    }
    return vec;
}

// Helper: Convert a std::vector<std::vector<float>> to a 2D numpy array (float32)
py::array_t<float> vector2d_to_numpy(const std::vector<std::vector<float>>& vec) {
    if (vec.empty())
        return py::array_t<float>();
    size_t k = vec.size();
    size_t d = vec[0].size();
    std::vector<ssize_t> shape = { static_cast<ssize_t>(k), static_cast<ssize_t>(d) };
    py::array_t<float> arr(shape);
    py::buffer_info buf = arr.request();
    float* ptr = static_cast<float*>(buf.ptr);
    for (size_t i = 0; i < k; i++) {
        for (size_t j = 0; j < d; j++) {
            ptr[i * d + j] = vec[i][j];
        }
    }
    return arr;
}

// AMPIIndex: Adaptive Multi-Projection Index
class AMPIIndex {
public:
    std::vector<std::vector<float>> data;         // Data: (n x d)
    int n, d, L;
    std::vector<std::vector<float>> proj_dirs;      // Projection directions: (L x d)
    std::vector<std::vector<float>> sorted_projs;   // Sorted projection values per direction: (L x n)
    std::vector<std::vector<int>> sorted_idxs;      // Corresponding sorted indices: (L x n)
    std::vector<float> data_norms;                  // Precomputed squared L2 norms for each data point.

    // Constructor: build index from data and number of projections.
    AMPIIndex(const std::vector<std::vector<float>>& data_in, int num_projections) {
        data = data_in;
        n = data.size();
        d = (n > 0) ? data[0].size() : 0;
        L = num_projections;

        // Precompute squared norms for each data point using BLAS.
        data_norms.resize(n, 0.0f);
        for (int i = 0; i < n; i++) {
            // Compute ||data[i]||^2 = data[i] · data[i]
            data_norms[i] = cblas_sdot(d, data[i].data(), 1, data[i].data(), 1);
        }

        // Initialize random generator with seed 0.
        std::mt19937 rng(0);
        std::normal_distribution<float> dist(0.0f, 1.0f);

        // Generate L random projection directions and normalize each.
        proj_dirs.resize(L, std::vector<float>(d, 0.0f));
        for (int i = 0; i < L; i++) {
            float norm = 0.0f;
            for (int j = 0; j < d; j++) {
                proj_dirs[i][j] = dist(rng);
                norm += proj_dirs[i][j] * proj_dirs[i][j];
            }
            norm = std::sqrt(norm);
            for (int j = 0; j < d; j++) {
                proj_dirs[i][j] /= norm;
            }
        }

        // Precompute projections for each direction.
        sorted_projs.resize(L, std::vector<float>(n, 0.0f));
        sorted_idxs.resize(L, std::vector<int>(n, 0));
        for (int i = 0; i < L; i++) {
            std::vector<float> projs(n, 0.0f);
            std::vector<int> idxs(n);
            for (int k = 0; k < n; k++) {
                float dot = 0.0f;
                for (int j = 0; j < d; j++) {
                    dot += data[k][j] * proj_dirs[i][j];
                }
                projs[k] = dot;
                idxs[k] = k;
            }
            // Sort indices by corresponding projection values.
            std::vector<int> sorted_indices = idxs;
            std::sort(sorted_indices.begin(), sorted_indices.end(), [&projs](int a, int b) {
                return projs[a] < projs[b];
            });
            // Save sorted projections and indices.
            for (int k = 0; k < n; k++) {
                sorted_projs[i][k] = projs[sorted_indices[k]];
                sorted_idxs[i][k] = sorted_indices[k];
            }
        }
    }

    // Extract candidate indices from projections given a query vector q.
    std::vector<int> query_candidates(const std::vector<float>& q, int window_size = 10) {
        std::vector<std::vector<int>> candidate_matrix(L, std::vector<int>(2 * window_size, 0));

        // Loop over each projection direction (parallelized if OpenMP is enabled)
#ifdef _OPENMP
        #pragma omp parallel for
#endif
        for (int i = 0; i < L; i++) {
            float q_val = 0.0f;
            for (int j = 0; j < d; j++) {
                q_val += proj_dirs[i][j] * q[j];
            }
            // Binary search in sorted_projs[i]
            auto& proj = sorted_projs[i];
            int idx = std::lower_bound(proj.begin(), proj.end(), q_val) - proj.begin();
            int slice_size = 2 * window_size;
            int start = idx - window_size;
            if (start < 0)
                start = 0;
            if (start > n - slice_size)
                start = n - slice_size;
            for (int k = 0; k < slice_size; k++) {
                candidate_matrix[i][k] = sorted_idxs[i][start + k];
            }
        }
        std::vector<int> flat;
        flat.reserve(L * 2 * window_size);
        for (int i = 0; i < L; i++) {
            for (int j = 0; j < 2 * window_size; j++) {
                flat.push_back(candidate_matrix[i][j]);
            }
        }
        std::sort(flat.begin(), flat.end());
        auto last = std::unique(flat.begin(), flat.end());
        flat.erase(last, flat.end());
        return flat;
    }

    // Compute squared Euclidean distances between query q and candidate points using BLAS.
    std::vector<float> compute_dists(const std::vector<float>& q, const std::vector<int>& candidate_indices) {
        std::vector<float> dists(candidate_indices.size(), 0.0f);
        // Precompute the squared norm of the query vector.
        float q_norm2 = cblas_sdot(d, q.data(), 1, q.data(), 1);
        for (size_t i = 0; i < candidate_indices.size(); i++) {
            int idx = candidate_indices[i];
            // Compute dot product between data[idx] and q using BLAS.
            float dot = cblas_sdot(d, data[idx].data(), 1, q.data(), 1);
            // Use the precomputed squared norm of data[idx] and the query.
            float dist2 = data_norms[idx] + q_norm2 - 2 * dot;
            dists[i] = dist2;
        }
        return dists;
    }

    // Query for k nearest neighbors.
    void query(const std::vector<float>& q, int window_size, int k,
               std::vector<std::vector<float>>& final_points,
               std::vector<float>& final_dists,
               std::vector<int>& final_indices) {
        std::vector<int> candidate_indices = query_candidates(q, window_size);
        std::vector<float> dists = compute_dists(q, candidate_indices);
        std::vector<int> order(candidate_indices.size());
        std::iota(order.begin(), order.end(), 0);
        std::sort(order.begin(), order.end(), [&dists](int a, int b) {
            return dists[a] < dists[b];
        });
        final_points.clear();
        final_dists.clear();
        final_indices.clear();
        for (int i = 0; i < k && i < (int)order.size(); i++) {
            int idx = candidate_indices[order[i]];
            final_points.push_back(data[idx]);
            final_dists.push_back(dists[order[i]]);
            final_indices.push_back(idx);
        }
    }

    // ---- Python Wrapper Methods ----

    std::vector<int> py_query_candidates(py::array_t<float> q, int window_size = 10) {
        std::vector<float> q_vec = numpy_to_vector1d(q);
        return query_candidates(q_vec, window_size);
    }

    py::tuple py_query(py::array_t<float> q, int window_size, int k) {
        std::vector<float> q_vec = numpy_to_vector1d(q);
        std::vector<std::vector<float>> final_points;
        std::vector<float> final_dists;
        std::vector<int> final_indices;
        query(q_vec, window_size, k, final_points, final_dists, final_indices);
        py::array_t<float> np_points = vector2d_to_numpy(final_points);
        return py::make_tuple(np_points, final_dists, final_indices);
    }
};

PYBIND11_MODULE(ampi_index, m) {
    m.doc() = "AMPIIndex module compiled with C++ and pybind11 (with BLAS for vectorized operations)";
    m.def("create_ampi_index", [](py::array_t<float> data, int num_projections) {
        auto vec = numpy_to_vector2d(data);
        return new AMPIIndex(vec, num_projections);
    }, py::return_value_policy::take_ownership, py::arg("data"), py::arg("num_projections"));

    py::class_<AMPIIndex>(m, "AMPIIndex")
        .def("query_candidates", &AMPIIndex::py_query_candidates, py::arg("q"), py::arg("window_size") = 10)
        .def("query", &AMPIIndex::py_query, py::arg("q"), py::arg("window_size"), py::arg("k"));
}


Overwriting ampi_index.cpp


In [5]:
!pip install pybind11 faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.3/243.3 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 19.3 MB/s eta 0:00:00


In [6]:
%%python3 --version

Python 3.11.11


In [7]:
!c++ -O3 -march=native -flto -Wall -shared -std=c++14 -fPIC -fopenmp $(python3.11 -m pybind11 --includes) ampi_index.cpp -o ampi_index$(python3.11-config --extension-suffix) -lopenblas


lto-wrapper: warning: using serial compilation of 4 LTRANS jobs


In [8]:
#!c++ -O3 -Wall -shared -std=c++14 -fPIC -fopenmp $(python3.11 -m pybind11 --includes) ampi_index.cpp -o ampi_index$(python3.11-config --extension-suffix)


In [9]:
#!c++ -O3 -Wall -shared -std=c++14 -fPIC $(python3.11 -m pybind11 --includes) ampi_index.cpp -o ampi_index$(python3.11-config --extension-suffix)

In [10]:
import numpy as np
import time
import faiss
from torchvision import datasets
import ampi_index  # Import the AMPI wrapper

def test_mnist(k=5, num_projections=128, window_size=128, num_queries=1_000):

    # Upload the dataset
    train_dataset = datasets.MNIST(root='./data', train=True, download=True)
    test_dataset  = datasets.MNIST(root='./data', train=False, download=True)

    # Flatten images and normalize pixel values.
    train_data = train_dataset.data.float().view(-1, 28*28) / 255.0
    test_data  = test_dataset.data.float().view(-1, 28*28) / 255.0
    X = np.concatenate([train_data, test_data], axis=0)  # shape: (70000, 784)
    n, d = X.shape

    # Build the FAISS index for exact L2 search.
    index_faiss = faiss.IndexFlatL2(d)
    index_faiss.add(X)

    # Build the AMPI index using the AMPI wrapper.
    ampi = ampi_index.create_ampi_index(X, num_projections)

    # Choose random queries.
    rng = np.random.default_rng(42)
    query_indices = rng.choice(n, size=num_queries, replace=False)

    error_ratios = []
    faiss_runtime = []
    ampi_runtime = []
    overlap_ratios = []  # List to store index overlap ratios

    for count, qi in enumerate(query_indices, start=1):
        q = X[qi]

        # FAISS exact k-NN search.
        start_faiss = time.perf_counter_ns()
        D_faiss, I_faiss = index_faiss.search(q.reshape(1, -1), k)
        end_faiss = time.perf_counter_ns()
        faiss_runtime.append(end_faiss - start_faiss)
        d_faiss = D_faiss[0]
        i_faiss = I_faiss[0]

        # AMPI approximate search.
        start_ampi = time.perf_counter_ns()
        approx_points, approx_dists, approx_indices = ampi.query(q, window_size, k)
        end_ampi = time.perf_counter_ns()
        ampi_runtime.append(end_ampi - start_ampi)
        d_ampi = np.array(approx_dists)
        i_ampi = np.array(approx_indices)

        print(f"Query {count}/{num_queries} (Index: {qi}):")
        print("  FAISS first neighbors -> distances:", d_faiss[:k], ", index:", i_faiss[:k])
        if len(d_ampi) > 0 and len(approx_indices) > 0:
            print("  AMPI first neighbors  -> distances:", d_ampi[:k], ", index:", approx_indices[:k])
        else:
            print("  AMPI returned no neighbors.")

        # Compute error ratio for the first neighbor, avoiding division by zero.
        ratio = np.divide(d_ampi, d_faiss, out=np.ones_like(d_ampi), where=(d_faiss != 0))
        error_ratios.append(np.mean(ratio))

        # Compute index overlap between FAISS and AMPI results.
        faiss_set = set(i_faiss[:k])
        ampi_set = set(approx_indices[:k])
        overlap = len(faiss_set.intersection(ampi_set))
        overlap_ratio = overlap / k
        overlap_ratios.append(overlap_ratio)
        print(f"  Index Overlap: {overlap} out of {k} (ratio: {overlap_ratio:.2f})\n")

    avg_ratio = np.mean(error_ratios)
    avg_faiss_time = np.mean(faiss_runtime)
    avg_ampi_time = np.mean(ampi_runtime)
    avg_overlap_ratio = np.mean(overlap_ratios)

    print(f"\nAverage index overlap ratio: {avg_overlap_ratio:.2f}")
    print(f"Average distance ratio (AMPI / FAISS): {avg_ratio:.6f}")
    print(f"FAISS average runtime per query: {avg_faiss_time/num_queries:.3f} ns")
    print(f"AMPI average runtime per query: {avg_ampi_time/num_queries:.3f} ns")

    tolerance = 1.10  # Allow up to 10% increase in distance.
    if avg_ratio > tolerance:
        print(f"AMPI's average distance ratio {avg_ratio:.6f} exceeds the tolerance {tolerance}")
    else:
        print(f"AMPI's average distance ratio {avg_ratio:.6f} is within the tolerance {tolerance}")

if __name__ == "__main__":
    # Use smaller numbers for quick testing; adjust parameters as needed.
    test_mnist(k=50, num_projections=512, window_size=128, num_queries=1000)

Streaming output truncated to the last 5000 lines.
 6.078585  6.186882  6.1939564 6.2174244 6.2195616 6.248566  6.419778
 6.4576087 6.504206  6.5501885 6.565859  6.587974  6.667774  6.6921034
 6.7316413 6.752126  6.8035684 6.8176703 6.83208   6.9092813 6.9230604
 7.009427  7.046136  7.0751863 7.099946  7.1701193 7.202553  7.3703036
 7.374825  7.383176  7.4325876 7.4523487 7.4625454 7.4652977 7.4883204
 7.5231676] , index: [64853 11297 19517 34074  2894 64524 39846 61884 33781 49932 24563 62693
 33861 20373 53610 21460 31144 46142 17029 31181 13510 69143 48079 34483
 39157 62529 25970 22462 60288  1174  3430 12842 26789 61214 37182 68658
 25687 65666 60790 50864  1012 10983 48085 43142 10885  6880 25531 26167
 29407   211]
  AMPI first neighbors  -> distances: [0.         4.05579376 4.16596985 4.54363251 5.2190094  5.85913086
 5.92565918 5.92849731 5.93618011 5.97715759 5.99314117 6.02648163
 6.05475616 6.06506348 6.07858276 6.18688965 6.19395447 6.21742249
 6.21955872 6.24856567 6.4197

In [11]:
import numpy as np
import time
import faiss
import ampi_index  # Import the AMPI wrapper

def test_randn(n=100_000, d=512, k=5, num_projections=128, window_size=128, num_queries=1_000):
    # Seed for reproducibility.
    np.random.seed(0)
    # Generate a random Gaussian dataset.
    X = np.random.randn(n, d).astype(np.float32)

    # Build the FAISS index for exact L2 search.
    index_faiss = faiss.IndexFlatL2(d)
    index_faiss.add(X)

    # Build the AMPI index using the AMPI wrapper.
    ampi = ampi_index.create_ampi_index(X, num_projections)

    # Choose random queries.
    rng = np.random.default_rng(42)
    query_indices = rng.choice(n, size=num_queries, replace=False)

    error_ratios = []
    faiss_runtime = []
    ampi_runtime = []
    overlap_ratios = []  # List to store index overlap ratios

    for count, qi in enumerate(query_indices, start=1):
        q = X[qi]

        # FAISS exact k-NN search.
        start_faiss = time.perf_counter_ns()
        D_faiss, I_faiss = index_faiss.search(q.reshape(1, -1), k)
        end_faiss = time.perf_counter_ns()
        faiss_runtime.append(end_faiss - start_faiss)
        d_faiss = D_faiss[0]
        i_faiss = I_faiss[0]

        # AMPI approximate search.
        start_ampi = time.perf_counter_ns()
        approx_points, approx_dists, approx_indices = ampi.query(q, window_size, k)
        end_ampi = time.perf_counter_ns()
        ampi_runtime.append(end_ampi - start_ampi)
        d_ampi = np.array(approx_dists)
        i_ampi = np.array(approx_indices)

        print(f"Query {count}/{num_queries} (Index: {qi}):")
        print("  FAISS first neighbors -> distances:", d_faiss[:k], ", index:", i_faiss[:k])
        if len(d_ampi) > 0 and len(approx_indices) > 0:
            print("  AMPI first neighbors  -> distances:", d_ampi[:k], ", index:", approx_indices[:k])
        else:
            print("  AMPI returned no neighbors.")

        # Compute error ratio for the first neighbor, avoiding division by zero.
        ratio = np.divide(d_ampi, d_faiss, out=np.ones_like(d_ampi), where=(d_faiss != 0))
        error_ratios.append(np.mean(ratio))

        # Compute index overlap between FAISS and AMPI results.
        faiss_set = set(i_faiss[:k])
        ampi_set = set(approx_indices[:k])
        overlap = len(faiss_set.intersection(ampi_set))
        overlap_ratio = overlap / k
        overlap_ratios.append(overlap_ratio)
        print(f"  Index Overlap: {overlap} out of {k} (ratio: {overlap_ratio:.2f})\n")

    avg_ratio = np.mean(error_ratios)
    avg_faiss_time = np.mean(faiss_runtime)
    avg_ampi_time = np.mean(ampi_runtime)
    avg_overlap_ratio = np.mean(overlap_ratios)

    print(f"\nAverage index overlap ratio: {avg_overlap_ratio:.2f}")
    print(f"Average distance ratio (AMPI / FAISS): {avg_ratio:.6f}")
    print(f"FAISS average runtime per query: {avg_faiss_time/num_queries:.3f} ns")
    print(f"AMPI average runtime per query: {avg_ampi_time/num_queries:.3f} ns")

    tolerance = 1.10  # Allow up to 10% increase in distance.
    if avg_ratio > tolerance:
        print(f"AMPI's average distance ratio {avg_ratio:.6f} exceeds the tolerance {tolerance}")
    else:
        print(f"AMPI's average distance ratio {avg_ratio:.6f} is within the tolerance {tolerance}")

if __name__ == "__main__":
    # Use smaller numbers for quick testing; adjust parameters as needed.
    test_randn(n=100_000, d=512, k=50, num_projections=1024, window_size=128, num_queries=1000)


Streaming output truncated to the last 5000 lines.
 870.50977 871.4209  871.9992  872.28876 872.6011  872.70935 872.7957
 872.9945  873.09863 873.8042  874.8065  875.2502  876.2814  876.3779
 876.47754] , index: [  546  7839 69288 32203 71345 80735 12383 65889 38995  3753 16579 29888
 70514 98879 82276 72661 67830 36023 75771 39826 21849 46759 93855 68203
 94640 14870 87516  4636  8719 81625 23923 14425  9783 97787 86719 97339
 55590 35869 46510 20330 17078 26227 12628 39542 28399 96956 75026 50973
 55842 26705]
  AMPI first neighbors  -> distances: [  0.         824.71051025 837.39227295 837.62744141 844.03717041
 849.51745605 850.47399902 850.9576416  852.43823242 853.56079102
 855.42480469 856.56530762 856.84197998 856.96130371 856.99920654
 857.09197998 858.59259033 858.64074707 858.64819336 859.73937988
 861.72869873 861.85559082 862.13757324 864.03991699 864.48529053
 864.53552246 864.61218262 864.80627441 864.8182373  865.26721191
 867.57055664 868.28161621 868.38214111 869.5642